# Procedural Memory

> Capture learned procedures, workflows, and "how-to" knowledge so your agent improves with experience.

Think about making coffee for the first time in a new kitchen. You open every cabinet, check every drawer, and fumble with the machine. By the tenth morning, your hands move on autopilot. You've internalized the procedure.

Most memory techniques store *what happened* (episodic) or *what is true* (semantic). Procedural memory stores *how to do things*. It captures the step-by-step strategies and workflows (sequences of actions that accomplish a goal) that an agent discovers through experience.

Here's the problem. When an agent solves a complex task through reasoning and tool calls, that successful action sequence is valuable knowledge. Without procedural memory, the agent re-derives the solution from scratch every time. With procedural memory, it retrieves the learned procedure and adapts it to the new situation.

Systems like Voyager (Wang et al., 2023) show this in practice. That Minecraft agent writes reusable code-based skills and stores them in a skill library (an organized collection of procedures). When a similar task appears, it retrieves the stored skill instead of starting over.

**In this notebook, you'll build a procedural memory system from scratch.** It will:
1. Extract reusable procedures from successful execution traces (logs of agent actions).
2. Store them in a skill library indexed by task type.
3. Retrieve matching procedures when new tasks arrive.
4. Adapt the retrieved procedure to the new context.

## Key Concepts

- **Execution trace**: A log of everything the agent did during a task. It includes actions taken, tools called, inputs provided, and outputs received.
- **Procedure extraction**: Identifying a reusable action sequence from a specific execution trace. An LLM (large language model, an AI that processes text) reviews the trace and distills it into a clean template.
- **Workflow template**: A parameterized procedure where specific values (file names, server addresses, app names) are replaced with placeholders like `{app_name}`. This makes the workflow apply to new inputs without rewriting it.
- **Skill library**: An organized collection of procedures indexed by task type. The library grows as the agent solves new kinds of tasks.
- **Procedure adaptation**: Filling in a template's placeholders with concrete values for a new task. The adapted procedure is ready to execute.
- **Procedure generalization**: Abstracting a specific procedure into a broader form. "Deploy weather-api to deploy.cloudhost.io" becomes "Deploy {app_name} to {server_host}".

## Architecture

<p align="center">
  <img src="../../images/diagrams/11_procedural_memory.svg" alt="Procedural Memory Architecture" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
    Task["Task Execution\n(step-by-step)"] --> SuccessCheck{"Task\nSucceeded?"}
    SuccessCheck -- No --> Discard["Discard / Log\nfor debugging"]
    SuccessCheck -- Yes --> Extractor["Procedure Extractor\n(LLM call)"]
    Extractor --> Template["Workflow Template\n(parameterized steps)"]
    Template --> Library["Skill Library\n(indexed by task type)"]
    NewTask["New Task"] --> Retrieval["Retrieval\n(task-type matching)"]
    Library --> Retrieval
    Retrieval --> Adaptation["Adaptation\n(fill parameters)"]
    Adaptation --> Execution["Execute Adapted\nProcedure"]
```

</details>

**Data flow:** When the agent completes a task, the execution trace goes to a procedure extractor (an LLM call). The extractor distills the action sequence into a parameterized workflow template. That template is stored in the skill library, indexed by task type and metadata.

When a new task arrives, the retrieval engine matches it against the library. If it finds a relevant procedure, it adapts the template by filling placeholders with concrete values. If no match exists, the agent solves the task from scratch. Then the cycle repeats.

## Setup

We need the OpenAI Python SDK for LLM calls and `python-dotenv` for loading API keys from a `.env` file.

In [ ]:
%pip install -q openai python-dotenv

Load the API key and create the OpenAI client. Set `OPENAI_API_KEY` in your `.env` file before running.

In [ ]:
import os
import json
from dataclasses import dataclass, field
from datetime import datetime

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()  # reads OPENAI_API_KEY from environment
MODEL = "gpt-4o-mini"

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

## Implementation

We'll build three components:

1. **Procedure data model.** A dataclass that holds the procedure's name, steps, parameters, and metadata.
2. **LLM prompt templates.** Three prompts that power extraction, retrieval, and adaptation.
3. **ProceduralMemory class.** The core engine that ties everything together.

### Procedure Data Model

Think of a recipe card in a box. Each card records the dish name, the category (appetizer, main, dessert), the ingredients you'll need, and the numbered steps. Our `Procedure` dataclass is that recipe card.

The `steps` field holds parameterized action descriptions. The `parameters` field lists the placeholder names used in those steps. The `preconditions` field records what must be true before the procedure can run.

In [ ]:
@dataclass
class Procedure:
    """A reusable, parameterized procedure extracted from a successful task."""

    name: str
    task_type: str
    description: str
    steps: list[str]           # parameterized action descriptions
    parameters: list[str]      # placeholder names (e.g., ["app_name", "server_host"])
    preconditions: list[str]   # what must be true before execution
    success_count: int = 0
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())

    def display(self) -> None:
        """Print a readable summary of this procedure."""
        print(f"Procedure: {self.name}")
        print(f"  Type: {self.task_type}")
        print(f"  Description: {self.description}")
        print(f"  Parameters: {self.parameters}")
        print(f"  Preconditions: {self.preconditions}")
        print(f"  Steps:")
        for i, step in enumerate(self.steps, 1):
            print(f"    {i}. {step}")
        print(f"  Success count: {self.success_count}")

### LLM Prompt Templates

The system makes three distinct LLM calls, each with its own prompt.

1. **Extraction prompt**: Reads an execution trace and produces a parameterized procedure.
2. **Retrieval prompt**: Compares a new task against the skill library and picks the best match.
3. **Adaptation prompt**: Fills in a procedure's placeholders with values from the new task context.

All three prompts request JSON output. This makes parsing reliable.

In [ ]:
EXTRACTION_PROMPT = """\
You are a procedure extractor. You receive an execution trace from a \
successful task. Your job: distill it into a reusable, parameterized procedure.

Rules:
- Replace specific values (file names, URLs, server addresses, app names) \
with descriptive placeholders like {app_name}, {server_host}, {file_path}.
- Keep the steps in execution order.
- Remove noise (debug output, retries, irrelevant side-effects).
- Identify preconditions (what must be true before this procedure can run).

Return a JSON object with these fields:
- name: short name for the procedure (e.g., "Deploy containerized app")
- task_type: category (e.g., "deployment", "data_pipeline", "api_integration")
- description: one-sentence summary of what this procedure accomplishes
- steps: list of parameterized step descriptions (strings)
- parameters: list of placeholder names used in the steps
- preconditions: list of conditions that must hold before execution
"""

RETRIEVAL_PROMPT = """\
You are a procedure retrieval engine. You receive a task description and a \
skill library (a list of stored procedures with index numbers).

Pick the best matching procedure for the task.

Return a JSON object with:
- best_match_index: integer index of the best match, or null if none is relevant
- confidence: float between 0.0 and 1.0
- reasoning: one sentence explaining your choice
"""

ADAPTATION_PROMPT = """\
You are a procedure adaptation engine. You receive a parameterized procedure \
and a new task context. Fill in the placeholders with concrete values from \
the context.

Return a JSON object with:
- adapted_steps: list of concrete step descriptions (placeholders filled in)
- parameter_values: object mapping each parameter name to its concrete value
"""

### ProceduralMemory Class

This class ties extraction, storage, retrieval, and adaptation together. The skill library is a Python list. In a production system, you'd use a database with vector search (finding similar items by comparing their numeric representations) for faster retrieval. For learning purposes, a list with LLM-based matching works well.

In [ ]:
class ProceduralMemory:
    """A skill library that learns procedures from successful task executions."""

    def __init__(self, client: OpenAI, model: str = "gpt-4o-mini"):
        self.client = client
        self.model = model
        self.skill_library: list[Procedure] = []

    # ── Extraction ───────────────────────────────────────────────
    def extract_procedure(self, execution_trace: str) -> Procedure:
        """Use the LLM to extract a reusable procedure from an execution trace."""
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": EXTRACTION_PROMPT},
                {"role": "user", "content": f"Execution trace:\n{execution_trace}"},
            ],
            response_format={"type": "json_object"},
        )
        data = json.loads(response.choices[0].message.content)
        return Procedure(
            name=data["name"],
            task_type=data["task_type"],
            description=data["description"],
            steps=data["steps"],
            parameters=data["parameters"],
            preconditions=data.get("preconditions", []),
        )

    # ── Storage ──────────────────────────────────────────────────
    def store_procedure(self, procedure: Procedure) -> str:
        """Add a procedure to the skill library. Merge if a similar one exists."""
        existing = self._find_similar(procedure)
        if existing:
            existing.success_count += 1
            return f"Merged with existing: '{existing.name}' (count: {existing.success_count})"
        self.skill_library.append(procedure)
        return f"Stored new procedure: '{procedure.name}'"

Next we add the retrieval and adaptation methods. Retrieval compares the new task description against all stored procedures using the LLM. Adaptation fills in a procedure's placeholders with concrete values from the new task context.

In [ ]:
    # ── Retrieval ────────────────────────────────────────────────
    def retrieve_procedure(self, task_description: str) -> tuple[Procedure | None, dict]:
        """Find the best matching procedure for a new task."""
        if not self.skill_library:
            return None, {"reasoning": "Skill library is empty."}

        library_summary = "\n".join(
            f"[{i}] {p.name} (type: {p.task_type}): {p.description}"
            for i, p in enumerate(self.skill_library)
        )

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": RETRIEVAL_PROMPT},
                {"role": "user", "content": (
                    f"Task: {task_description}\n\n"
                    f"Skill library:\n{library_summary}"
                )},
            ],
            response_format={"type": "json_object"},
        )
        result = json.loads(response.choices[0].message.content)
        idx = result.get("best_match_index")
        if idx is not None and 0 <= idx < len(self.skill_library):
            return self.skill_library[idx], result
        return None, result

    # ── Adaptation ───────────────────────────────────────────────
    def adapt_procedure(self, procedure: Procedure, task_context: str) -> dict:
        """Adapt a procedure's steps to a new context by filling placeholders."""
        steps_text = "\n".join(
            f"  {i+1}. {s}" for i, s in enumerate(procedure.steps)
        )

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": ADAPTATION_PROMPT},
                {"role": "user", "content": (
                    f"Procedure: {procedure.name}\n"
                    f"Steps:\n{steps_text}\n\n"
                    f"Parameters: {procedure.parameters}\n\n"
                    f"New task context: {task_context}"
                )},
            ],
            response_format={"type": "json_object"},
        )
        return json.loads(response.choices[0].message.content)

The `learn_from_execution` method ties the loop together: if a task succeeded, it extracts and stores a procedure in one call. The helper methods handle duplicate detection and library listing.

In [ ]:
    # ── Full learning loop ───────────────────────────────────────
    def learn_from_execution(
        self, execution_trace: str, success: bool
    ) -> Procedure | None:
        """Extract and store a procedure if the task succeeded."""
        if not success:
            print("Task failed. Skipping procedure extraction.")
            return None
        procedure = self.extract_procedure(execution_trace)
        status = self.store_procedure(procedure)
        print(status)
        return procedure

    # ── Internal helpers ─────────────────────────────────────────
    def _find_similar(self, procedure: Procedure) -> Procedure | None:
        """Check if a similar procedure already exists (by type and name)."""
        for p in self.skill_library:
            if (
                p.task_type == procedure.task_type
                and p.name.lower() == procedure.name.lower()
            ):
                return p
        return None

    def list_procedures(self) -> None:
        """Print all procedures in the skill library."""
        if not self.skill_library:
            print("Skill library is empty.")
            return
        for i, p in enumerate(self.skill_library):
            print(f"\n{'='*60}")
            print(f"[{i}] ", end="")
            p.display()

## Example Run

Let's see procedural memory in action. We'll walk through three scenarios:

1. An agent deploys a web API. We extract and store the deployment procedure.
2. A similar deployment task arrives. We retrieve the stored procedure and adapt it.
3. A data pipeline task runs. We extract a second procedure, growing the library.

### Scenario 1: Learn a Deployment Procedure

Below is an execution trace from an agent that deployed a Flask API to a cloud server. Each line records one action the agent took.

In [ ]:
deployment_trace = """\
1. Created project directory "weather-api"
2. Initialized Python 3.11 virtual environment in weather-api/venv
3. Installed dependencies: flask, requests, gunicorn (pip install)
4. Wrote application code in weather-api/app.py with /forecast and /health endpoints
5. Wrote unit tests in weather-api/test_app.py
6. Ran pytest on test_app.py - all 8 tests passed
7. Created Dockerfile with python:3.11-slim base image
8. Built Docker image tagged "weather-api:v1"
9. Pushed image to container registry at registry.cloudhost.io/weather-api:v1
10. Connected via SSH to server at deploy.cloudhost.io
11. Pulled the Docker image on the server
12. Started container with port mapping 80:5000 using docker-compose
13. Verified health check at https://deploy.cloudhost.io/health returned 200 OK
"""

print(deployment_trace)

We pass the trace to `learn_from_execution`. The LLM extracts a parameterized procedure and the system stores it in the skill library.

In [ ]:
memory = ProceduralMemory(client=client, model=MODEL)

procedure = memory.learn_from_execution(deployment_trace, success=True)
print()
procedure.display()

### Scenario 2: Retrieve and Adapt for a New Task

A new task arrives: deploy a different API to a different server. Instead of solving from scratch, the agent checks its skill library first.

In [ ]:
new_task = "Deploy the inventory-service (a FastAPI app) to production server prod.mycompany.com"

matched_procedure, retrieval_info = memory.retrieve_procedure(new_task)

print("Retrieval result:")
print(f"  Matched: {matched_procedure.name if matched_procedure else 'None'}")
print(f"  Confidence: {retrieval_info.get('confidence', 'N/A')}")
print(f"  Reasoning: {retrieval_info.get('reasoning', 'N/A')}")

The retrieval engine found our deployment procedure. Now we adapt it. The adaptation step fills in each placeholder with a concrete value from the new task.

In [ ]:
task_context = (
    "App name: inventory-service. "
    "Framework: FastAPI. "
    "Server: prod.mycompany.com. "
    "Registry: docker.mycompany.com. "
    "Port mapping: 80:8000. "
    "Python version: 3.12."
)

adaptation = memory.adapt_procedure(matched_procedure, task_context)

print("Adapted steps for the new task:\n")
for i, step in enumerate(adaptation["adapted_steps"], 1):
    print(f"  {i}. {step}")

print(f"\nParameter values:")
print(json.dumps(adaptation.get("parameter_values", {}), indent=2))

### Scenario 3: Grow the Library

Now the agent runs a data pipeline task. This is a different task type, so the system creates a new procedure alongside the deployment one.

In [ ]:
pipeline_trace = """\
1. Connected to PostgreSQL database at db.analytics.io:5432
2. Queried the 'sales' table for records where date >= '2024-01-01' and date < '2024-02-01'
3. Loaded 15,420 rows into a pandas DataFrame
4. Cleaned data: dropped 23 rows with null revenue, converted currency columns to USD
5. Grouped by region and product_category, computed sum of revenue and count of orders
6. Generated a summary CSV report with 48 rows (one per region-category pair)
7. Uploaded report to S3 at s3://company-reports/monthly/2024-01-sales-summary.csv
8. Sent notification email to analytics-team@company.com with the S3 link
"""

pipeline_procedure = memory.learn_from_execution(pipeline_trace, success=True)

The skill library now holds two procedures from different domains. Let's inspect both.

In [ ]:
memory.list_procedures()

### Selective Retrieval

With multiple procedures in the library, the retrieval engine must pick the right one. Let's test it with tasks from each domain.

In [ ]:
# A deployment task should match the deployment procedure
task_a = "Deploy the user-auth microservice to staging server staging.myapp.io"
match_a, info_a = memory.retrieve_procedure(task_a)
print(f"Task: {task_a}")
print(f"  Matched: {match_a.name if match_a else 'None'}")
print(f"  Reasoning: {info_a.get('reasoning', 'N/A')}")

print()

# A data task should match the pipeline procedure
task_b = "Generate the February 2024 sales report from the analytics database"
match_b, info_b = memory.retrieve_procedure(task_b)
print(f"Task: {task_b}")
print(f"  Matched: {match_b.name if match_b else 'None'}")
print(f"  Reasoning: {info_b.get('reasoning', 'N/A')}")

## Tradeoffs

### When Procedural Memory Works Well

- **Repetitive multi-step tasks.** Agents that deploy services, run data pipelines, or integrate APIs perform similar workflows repeatedly. Storing the procedure avoids re-deriving the steps each time.
- **Skill transfer across sessions.** A procedure extracted in one session is available in the next. The agent's capability grows over time without retraining or fine-tuning.
- **Faster execution.** Retrieving a stored procedure and adapting it takes fewer LLM calls than solving from scratch with multi-step reasoning.

### When It Breaks Down

- **Novel tasks.** If every task is unique, the skill library never gets reused. The extraction cost adds overhead with no payoff.
- **Fragile procedures.** A procedure extracted from one successful run may not generalize well. If the environment changes (different OS, different API version), the stored steps can fail without warning. You need feedback loops and versioning to handle this.
- **Retrieval errors.** LLM-based retrieval can match the wrong procedure, especially as the library grows large. Production systems need confidence thresholds and fallback behavior (solve from scratch if confidence is low).
- **Stale procedures.** Procedures become outdated as tools and APIs change. Without a way to invalidate or update old procedures, the library accumulates unreliable knowledge.

## Further Reading

- [Wang et al., "Voyager: An Open-Ended Embodied Agent with Large Language Models," 2023](https://arxiv.org/abs/2305.16291) - A Minecraft agent that builds a reusable skill library of code-based procedures. The closest existing system to what we built here.
- [Sumers et al., "Cognitive Architectures for Language Agents," 2023](https://arxiv.org/abs/2309.02427) - A survey of cognitive architectures for LLM agents. Covers procedural memory as a core component alongside episodic and semantic memory.
- [Anderson, "Acquisition of Cognitive Skill," *Psychological Review*, 1982](https://doi.org/10.1037/0033-295X.89.4.369) - The foundational psychology paper on how humans transition from declarative knowledge ("knowing that") to procedural knowledge ("knowing how").
- [Park et al., "Generative Agents: Interactive Simulacra of Human Behavior," 2023](https://arxiv.org/abs/2304.03442) - Simulated agents that develop behavioral patterns through memory and reflection. A form of emergent procedural knowledge.

---

*\u2190 Previous: [10 - Semantic Memory](../10_semantic_memory/) \u00b7 Next: [12 - Working Memory / Context Window](../12_working_memory_context_window/) \u2192*

## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: Procedure versioning
Add a `version` field and a `history` list to the `Procedure` dataclass. Each time `store_procedure()` updates an existing procedure, increment the version and archive the previous one. Print the version history after several learning cycles.

### Challenge 2: Adaptation success rate
Create 5 task variants that require adapting the same base procedure. Run `adapt_procedure()` for each, then execute the adapted procedure and score success. Report the adaptation success rate and identify which types of adaptations fail most often.

### Challenge 3: Memory-management procedures
Use `ProceduralMemory` to store procedures about when to summarize, when to forget, and when to consolidate memories. Feed these procedures back into an agent that manages its own memory. This combines procedural memory with the routing logic from 17 Memory Routing.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--11-procedural-memory--procedural-memory)